In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [5]:
df_20182021 = pd.read_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_cleaned_2018_2021.csv", thousands=",")
df_20212026 = pd.read_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_cleaned_2021_2026.csv", thousands=",")

C:\Users\gelet\AppData\Local\Temp\ipykernel_23404\873818448.py:1: DtypeWarning: Columns (32: search_types, 41: legal_basises_relevant_factors, 42: subject_restrained, 43: subject_restraint_devices) have mixed types. Specify dtype option on import or set low_memory=False.
  df_20182021 = pd.read_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_cleaned_2018_2021.csv", thousands=",")


In [5]:
print(df_20182021.shape)
print(df_20212026.shape)

(148205, 47)
(86729, 47)


In [6]:
# Change time variable into datetime from string
df_20182021["EventDate"] = df_20182021["EventDate"].str.strip()
df_20182021["EventDate"] =  pd.to_datetime(df_20182021["EventDate"], format="%Y %b %d %I:%M:%S %p", errors="coerce")

df_20212026["EventDate"] = df_20212026["EventDate"].str.strip()
df_20212026["EventDate"] =  pd.to_datetime(df_20212026["EventDate"], format="%Y %b %d %I:%M:%S %p", errors="coerce")

In [15]:
# Combine dataframes
df = pd.concat([df_20182021, df_20212026], ignore_index=True)

In [16]:
# Add extra date variables
df["month"] = df["EventDate"].dt.month
df["year"] = df["EventDate"].dt.year
df["weekend"] = (df["EventDate"].dt.weekday >= 5).astype(int)

In [17]:
# Only examine stops on or after March 1, 2018
df = df[(df["month"] >= 3) & (df["year"] >= 2018)]

In [18]:
# Only examine vehicle stops
df = df[(df["subject_type"] == "Passenger") | (df["subject_type"] == "Driver")]

In [19]:
# Recoding some categorical variables
df["SubjectGender"] = df["SubjectGender"].map({"FEMALE": 1, "MALE": 0})
df["SubjectHasPhotoID"] = df["SubjectHasPhotoID"].map({"Yes": 1, "No": 0})
df["isDriver"] = df["subject_type"].map({"Driver":1, "Passenger":0})
df["search_occurred"] = df["search_occurred"].map({"Yes": 1, "No": 0})
df['isLA'] = (df["SubjectDriverLicState"] == 'LA').astype('Int64')
df.loc[df["SubjectDriverLicState"].isna(), 'isLA'] = pd.NA

In [20]:
# Dropping post search variables
# We cannot use these variables as predictors because they may/will only happen after a search already occurs
df = df.drop(["evidence_seized", "search_types", "consent_to_search", "pat_down", "consent_form_completed", "stripbody_cavity_search", "legal_basises", "legal_basises_relevant_factors", "stop_results", "subject_restrained", "subject_restraint_devices", "exit_vehicle", "evidence_types"], axis=1)

# Dropping redundant variables
df = df.drop(["EventDate", "ActionsTaken", "CreatedDateTime", "LastModifiedDateTime", "subject_type"], axis=1)

In [21]:
df

,FieldInterviewID,NOPD_Item,District,Zone,OfficerAssignment,StopDescription,VehicleYear,VehicleMake,VehicleModel,VehicleStyle,...,BlockAddress,search_occurred,event_dt,hour,night,month,year,weekend,isDriver,isLA
1,571395,J0050921,4,E,4th District,TRAFFIC VIOLATION,2009.0,HONDA,ODYSSEY,FOUR DOOR,...,Sullen Pl & Woodland Hwy,0,2021-10-01 22:05:00,22,1,10,2021,0,1,1
2,571394,J0115921,7,F,7th District,TRAFFIC VIOLATION,2009.0,LEXUS,OTHER,FOUR DOOR,...,Chef Menteur Hwy & Dodt Ave,1,2021-10-01 21:37:00,21,1,10,2021,0,1,<NA>
3,571393,J0112821,3,J,ISB,TRAFFIC VIOLATION,2011.0,FORD,FOCUS,FOUR DOOR,...,036XX Canal St,0,2021-10-01 20:38:00,20,1,10,2021,0,1,1
4,571391,J0110421,7,O,7th District,TRAFFIC VIOLATION,2018.0,MERCEDES-BENZ,OTHER,FOUR DOOR,...,Crowder Blvd & N I-10 Service Rd,0,2021-10-01 20:27:00,20,1,10,2021,0,1,1
5,571392,J0104121,7,F,7th District,TRAFFIC VIOLATION,2013.0,DODGE,CHARGER,FOUR DOOR,...,Crowder Blvd & Chef Menteur Hwy,0,2021-10-01 20:06:00,20,1,10,2021,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234919,571408,J0167721,4,J,Traffic,TRAFFIC VIOLATION,2010.0,CADILLAC,CTS,FOUR DOOR,...,056XX General De Gaulle Dr,0,2021-10-02 09:13:00,9,0,10,2021,1,1,1
234920,572608,J3007721,4,B,Traffic,TRAFFIC VIOLATION,2004.0,TOYOTA,TACOMA,PICK UP,...,012XX L B Landry Ave,0,2021-10-02 08:13:00,8,0,10,2021,1,1,<NA>
234922,571402,J0150921,5,C,5th District,TRAFFIC VIOLATION,2004.0,CHEVROLET,SILVERADO,PICK UP,...,011XX Franklin Ave,0,2021-10-02 04:39:00,4,1,10,2021,1,1,1
234924,571401,J0145621,5,B,5th District,CALL FOR SERVICE,NaN,NaN,NaN,NaN,...,Saint Claude Ave & Spain St,0,2021-10-02 04:12:00,4,1,10,2021,1,1,0


In [22]:
# Splitting data into test and train
# stratify is used to ensure the proportion of searches is the same in the test and train data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=10, stratify=df["search_occurred"])

In [23]:
train_df

,FieldInterviewID,NOPD_Item,District,Zone,OfficerAssignment,StopDescription,VehicleYear,VehicleMake,VehicleModel,VehicleStyle,...,BlockAddress,search_occurred,event_dt,hour,night,month,year,weekend,isDriver,isLA
25709,542115,H2165220,3,D,3rd District,SUSPECT VEHICLE,2014.0,DODGE,OTHER,FOUR DOOR,...,Paris Ave & Westbrook Dr,1,2020-08-18 22:43:00,22,1,8,2020,0,1,1
167837,667644,D1779325,5,K,5th District,TRAFFIC VIOLATION,2007.0,ACURA,MDX,SPORTS UTILITY,...,St Bernard Av & N Broad St,0,2025-04-18 21:59:00,21,1,4,2025,0,1,1
208474,617735,G0706923,4,D,4th District,TRAFFIC VIOLATION,2022.0,TOYOTA,RAV4,SPORTS UTILITY,...,Holiday Dr & General De Gaulle Dr,0,2023-07-08 04:08:00,4,1,7,2023,1,1,0
93928,482625,K2726018,3,U,3rd District,TRAFFIC VIOLATION,2005.0,HONDA,ACCORD,TWO DOOR,...,Chef Menteur Hwy & Press Dr,0,2018-11-20 20:28:00,20,1,11,2018,0,0,<NA>
127149,452319,E2804318,6,M,Traffic,TRAFFIC VIOLATION,2011.0,HYUNDAI,SONATA,NaN,...,S Broad St & Erato St,0,2018-05-22 00:12:00,0,1,5,2018,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45329,524702,J0965819,7,C,7th District,SUSPECT VEHICLE,2009.0,CHEVROLET,OTHER,SPORTS UTILITY,...,Stemway Dr & Ransom St,0,2019-10-07 20:44:00,20,1,10,2019,0,1,0
159726,677968,G1321025,3,E,3rd District,TRAFFIC VIOLATION,2015.0,BUICK,OTHER,SPORTS UTILITY,...,Harrison Av & Catina St,0,2025-07-13 20:33:00,20,1,7,2025,1,1,1
192011,643239,F0443324,6,H,6th District,SUSPECT VEHICLE,2007.0,CHEVROLET,TRAILBLAZER,SPORTS UTILITY,...,Martin Luther King Jr Blvd & S Liberty St,1,2024-06-05 15:15:00,15,0,6,2024,0,1,1
169846,665855,C2350425,3,L,Traffic,TRAFFIC VIOLATION,2012.0,AUDI,OTHER,FOUR DOOR,...,034XX De Saix Blvd,0,2025-03-25 09:27:00,9,0,3,2025,0,1,<NA>


In [24]:
train_df.to_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_train1.csv", index=False)
test_df.to_csv(r"C:\Users\gelet\PycharmProjects\stop-and-search\data\cleaned_data\stop_search_test1.csv", index=False)

In [25]:
print(sum(test_df["search_occurred"]))
print(sum(train_df["search_occurred"]))

1670
6681
